<a href="https://colab.research.google.com/github/FarabiOnAMission/Machine-Learning-Stuffs/blob/main/Vectors_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
import math

In [10]:
class Vector:
  def __init__(self, components):
    self.components = list(components)
    self.dim = len(self.components)

  def __add__(self, other):
    return Vector(x + y for x,y in zip(self.components, other.components))

  def __sub__(self, other):
    return Vector(x - y for x,y in zip(self.components, other.components))

  def dot(self, other):
    return sum(x * y for x,y in zip(self.components, other.components))

  def magnitude(self):
    return sum(x**2 for x in self.components)**0.5

  def normalize(self):
    mag = self.magnitude()
    return Vector(x/mag for x in self.components)

  def cosine_similarity(self,other):
    return self.dot(other)/(self.magnitude()*other.magnitude())

  def __repr__(self):
    return f"Vector({self.components})"

  def __angle_between__(self, other):
    return math.acos(self.cosine_similarity(other))

In [11]:
a = Vector([1,2,3])
b = Vector([4,5,6])

print(f"a + b = {a+b}")
print(f"a - b = {a-b}")
print(f"a dot b = {a.dot(b)}")

a + b = Vector([5, 7, 9])
a - b = Vector([-3, -3, -3])
a dot b = 32


In [28]:
matrix = np.array([[4,2],[1,3]],dtype = float)
eigenvalues , eigenvector = np.linalg.eig(matrix)
print(f"Eigenvalues: {eigenvalues}")
print(f"Eigenvectors: {eigenvector}")


Eigenvalues: [5. 2.]
Eigenvectors: [[ 0.89442719 -0.70710678]
 [ 0.4472136   0.70710678]]


In [12]:
class Matrix:
    def __init__(self, rows):
        self.rows = [list(row) for row in rows ]
        self.shape = (len(self.rows), len(self.rows[0]))
    def __matmul__(self, other):
          if isinstance(other, Vector):
            return Vector([sum(self.rows[i][j] * other.components[j] for j in range(self.shape[1]))
                   for i in range(self.shape[0])])
          rows = []

          for i in range(self.shape[0]):
            row = []
            for j in range(other.shape[1]):
              row.append(sum(self.rows[i][k] * other.rows[k][j] for k in range(other.shape[0])
              ))
            rows.append(row)
          return Matrix(rows)
    def transpose(self):
      return Matrix([
          [self.rows[j][i] for j in range(self.shape[0])]
          for i in range(self.shape[1])
      ])

    def __repr__(self):
      return f"Matrix({self.rows})"

    def determinant(self):
      if self.shape[0] != self.shape[1]:
        raise ValueError("Matrix must be square")
      if self.shape[0] == 1:
        return self.rows[0][0]
      elif self.shape[0] == 2:
        return self.rows[0][0] * self.rows[1][1] - self.rows[0][1] * self.rows[1][0]
      det = 0
      for j in range(self.shape[0]):
        submatrix = [
            [self.rows[i][k] for k in range(self.shape[0]) if k != j]
            for i in range(1, self.shape[0])
        ]
        submatrix = Matrix(submatrix)
        det += (-1) ** j * self.rows[0][j] * submatrix.determinant()
      return det

    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix([
            [self.rows[1][1] / det, -self.rows[0][1] / det],
            [-self.rows[1][0] / det, self.rows[0][0] / det]
        ])

    def inverse_3x3(self):
      if self.shape[0] != self.shape[1]:
        raise ValueError("Matrix must be square")

      det = self.determinant()

      if abs(det) < 1e-10:
        raise ValueError("Matrix is singular, no inverse exists")

      if self.shape[0] == 1:
        return Matrix([[1/self.rows[0][0]]])

      cofactors = []
      for i in range(self.shape[0]):
        row = []
        for j in range(self.shape[1]):
          submatrix = [
              [self.rows[k][l] for l in range(self.shape[1]) if l!=j]
            for k in range(self.shape[0]) if k!=i]
          submatrix = Matrix(submatrix)
          cofactor = (-1)**(i+j) * submatrix.determinant()
          row.append(cofactor)
        cofactors.append(row)
      cofactors = Matrix(cofactors)
      adjugate = cofactors.transpose()

      return Matrix([[elem / det for elem in row] for row in adjugate.rows])




In [13]:
import math

def rotation_2d(theta):
  cs = math.cos(theta)
  sn = math.sin(theta)
  return Matrix([[cs, -sn], [sn, cs]])

def scaling_2d(sx, sy):
  return Matrix([[sx,0],[0,sy]])

def shearing_2d(kx,ky):
  return Matrix([[1,ky],[1,kx]])

def reflection_x():
  return Matrix([[1,0],[0,-1]])

def reflection_y():
  return Matrix([[-1,0],[0,1]])

def reflection_xy():
  return Matrix([[-1,0],[0,-1]])

def mat_vec_mul(matrix, vector):
    return [
        sum(matrix[i][j] * vector[j] for j in range(len(vector)))
        for i in range(len(matrix))
    ]

def mat_mul(a, b):
    rows_a, cols_b = len(a), len(b[0])
    cols_a = len(a[0])
    return [
        [sum(a[i][k] * b[k][j] for k in range(cols_a)) for j in range(cols_b)]
        for i in range(rows_a)
    ]

point = Vector([1.0, 0.0])
angle = math.pi / 4

rotation_matrix = rotation_2d(angle)
rotated_point = rotation_matrix @ point
print(f"Original: {point}")
print(f"Rotated: {rotated_point}")

Original: Vector([1.0, 0.0])
Rotated: Vector([0.7071067811865476, 0.7071067811865475])


In [26]:
theta = math.pi / 2
scaling_matrix = scaling_2d(2, 3)
shearing_matrix = shearing_2d(1, 2)
rotation_matrix = rotation_2d(theta)
corner1 = Vector([0,0])
corner2 = Vector([1,0])
corner3 = Vector([0,1])
corner4 = Vector([1,1])

print(f"Original: {corner1}, {corner2}, {corner3}, {corner4}")
print(f"Rotated: {rotation_matrix @ corner1}, {rotation_matrix @ corner2}, {rotation_matrix @ corner3}, {rotation_matrix @ corner4}")
print(f"Scaled: {scaling_matrix @ corner1}, {scaling_matrix @ corner2}, {scaling_matrix @ corner3}, {scaling_matrix @ corner4}")
print(f"Sheared: {shearing_matrix @ corner1}, {shearing_matrix @ corner2}, {shearing_matrix @ corner3}, {shearing_matrix @ corner4}")

Original: Vector([0, 0]), Vector([1, 0]), Vector([0, 1]), Vector([1, 1])
Rotated: Vector([0.0, 0.0]), Vector([6.123233995736766e-17, 1.0]), Vector([-1.0, 6.123233995736766e-17]), Vector([-0.9999999999999999, 1.0])
Scaled: Vector([0, 0]), Vector([2, 0]), Vector([0, 3]), Vector([2, 3])
Sheared: Vector([0, 0]), Vector([1, 1]), Vector([2, 1]), Vector([3, 2])


In [14]:

rotation_90 = Matrix([[0, -1], [1, 0]])
point = Vector([3, 1])

rotated = rotation_90 @ point
print(f"Original: {point}")
print(f"Rotated 90°: {rotated}")

Original: Vector([3, 1])
Rotated 90°: Vector([-1, 3])


In [22]:
def eigenvalues_2x2(matrix):
  a, b = matrix[0]
  c, d = matrix[1]

  det = a * d - b * c
  trace = a + d
  discriminant = trace ** 2 - 4 * det
  if discriminant < 0:
    real = trace/2
    img = (-discriminant) ** 0.5 / 2
    return (complex(real,img),complex(real,-img))
  else:
    return ((trace + discriminant ** 0.5)/2, (trace - discriminant ** 0.5)/2)

def eigenvector_2x2(matrix, eigenvalue):
    a, b = matrix[0]
    c, d = matrix[1]
    if abs(b) > 1e-10:
        v = [b, eigenvalue - a]
    elif abs(c) > 1e-10:
        v = [eigenvalue - d, c]
    else:
        if abs(a - eigenvalue) < 1e-10:
            v = [1, 0]
        else:
            v = [0, 1]
    mag = (v[0] ** 2 + v[1] ** 2) ** 0.5
    return [v[0] / mag, v[1] / mag]

In [24]:
A = [[2,1],[1,2]]
vals = eigenvalues_2x2(A)
print(f"Eigenvalues: {vals}")

for val in vals:
  vec = eigenvector_2x2(A,val)
  result = mat_vec_mul(A,vec)
  print(f"Eigenvector for {val}: {vec}")
  scaled = [val * vec[0], val * vec[1]]
  print(f"  lambda={val:.1f}, v={[round(x,4) for x in vec]}")
  print(f"    A@v = {[round(x,4) for x in result]}")
  print(f"    l*v = {[round(x,4) for x in scaled]}")


Eigenvalues: (3.0, 1.0)
Eigenvector for 3.0: [0.7071067811865475, 0.7071067811865475]
  lambda=3.0, v=[0.7071, 0.7071]
    A@v = [2.1213, 2.1213]
    l*v = [2.1213, 2.1213]
Eigenvector for 1.0: [0.7071067811865475, -0.7071067811865475]
  lambda=1.0, v=[0.7071, -0.7071]
    A@v = [0.7071, -0.7071]
    l*v = [0.7071, -0.7071]


In [15]:
def _is_linearly_independent(vectors):
  #vectors = [v1,v2,v3]
  # v1 = [x1,x2,x3]
  n = len(vectors)
  #counts how many vectors in the matrix
  dim = len(vectors[0].components)
  #features of each vector
  rows = [v.components[:] for v in vectors]

  """
   [
      [x1,x2,x3],
      [y1,y2,y3],
      [z1,z2,z3]
  ]
  """
  rank = 0
  #current rank

  #its a Row Matrix meaning each row contains a vector
  #Because row is easily accessible in python instead of columns

  for col in range(dim):
    #loops through each column
    pivot = None
    #looping through all rows of the current column
    for row in range(rank, len(rows)):
      if abs(rows[row][col]) > 1e-10:
        #finds the first non-zero row
        #then marks it as pivot and breaks
        pivot = row
        break
    if pivot is None:
      continue

    rows[rank], rows[pivot] = rows[pivot], rows[rank]
    #swaps the found pivot row with the rank row

    scale = rows[rank][col]

    #To make the pivot element 1 we divide the whole row by this element

    rows[rank] = [x/scale for x in rows[rank]]
    for row in range(len(rows)):
      if row != rank and abs(rows[row][col]) >1e-10:
        factor = rows[row][col]
        rows[row] = [rows[row][j] - factor * rows[rank][j] for j in range(dim)]
      #subtracts each element in the pivot column and makes it zero so theres only one row
    rank += 1

  return rank


def project(a,b):
  scalar = a.dot(b)/b.dot(b)
  return Vector([scalar * x for x in b.components])


def grahm_schimdt(vectors):
  orthonormal = []
  for v in vectors:
    w = v
    for u in orthonormal:
      w = w - project(v,u)
    orthonormal.append(w)
  return orthonormal

In [16]:
#Scale Matrix , which scales the x factor by 2 and scales the y factor by 3
twoD_Rows = [[2,0],[0,3]]
Scale_Matrix = Matrix(twoD_Rows)
print(f"Scale Matrix: {Scale_Matrix}")

new_Vector = Vector([1,1])
print(f"Original: {new_Vector}")
print(f"Scaled: {Scale_Matrix @ new_Vector}")

Scale Matrix: Matrix([[2, 0], [0, 3]])
Original: Vector([1, 1])
Scaled: Vector([2, 3])


In [17]:
#Calculates the max similarity out of 5 vectors with 10 compoponents
vector_rows = [Vector(np.random.rand(50)) for _ in range(5)]
max_similarity = -1.1

for i in range(len(vector_rows)):
  for j in range(i+1,len(vector_rows)):
    similarity = vector_rows[i].cosine_similarity(vector_rows[j])
    max_similarity = max(max_similarity,similarity)

print(f"Max Similarity: {max_similarity}")


Max Similarity: 0.7576942758303584


In [18]:
#Checking if a matrix has rank 2
v1 = Vector([0, 1, 0])
v2 = Vector([1, 0, 0])
v3 = Vector([2, 1, 0])

matrix_vectors = [v1, v2, v3]

calculated_rank = _is_linearly_independent(matrix_vectors)

print(calculated_rank)

2
